# Textual Irony Detection (`UmBERTo`) — Two-Stage Pre-training & Controlled Correlation Ablation
**Task:** Single-Label 3-Class Classification (`None` = 0, `Ironia` = 1, `Sarcasmo` = 2)  
**Base Encoder:** `Musixmatch/umberto-commoncrawl-cased-v1`  
**Pipeline Design:**
1. **Stage 0 (Shared Base Pre-training on `IronITA`):** Pre-trains `UmBERTo` on `ironita_full.xlsx` (4,849 Italian tweets with 3-class labels `None`, `Ironia`, `Sarcasmo`) and saves `./checkpoints/irony_textual_ironita_pretrained`.
2. **Stage 1A (MultiEMit Without Correlations):** Fine-tunes the shared `IronITA`-pretrained checkpoint on the 80% train split of `MultiEMit_textual_no_incomprensibile.csv` (raw text) and evaluates on the 20% test split (`seed=42`).
3. **Stage 1B (MultiEMit With Correlations):** Fine-tunes the **exact same** shared `IronITA`-pretrained checkpoint on the **exact same** 80% train split of `MultiEMit_textual_no_incomprensibile.csv` augmented with full Italian correlation prefixes (`Emotions` + `Target`) and evaluates on the 20% test split.

> **Checkpoint Resume / Inference-Only Mode:** By default, `FORCE_RETRAIN = False`. If a stage's checkpoint already exists in `./checkpoints/`, training is skipped automatically and the saved model is loaded directly for evaluation.


In [1]:
# Install CUDA-enabled PyTorch (supports GTX 1080 Ti / Pascal via CUDA 12.1) and all requirements
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
%pip install -q -r ../requirements.txt


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ['TQDM_NOTEBOOK'] = 'false'
import random
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)
import tqdm.auto, tqdm.std
tqdm.auto.tqdm = tqdm.std.tqdm  # Force plain-text progress bars in VS Code

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cuda


## 1. Configuration & Dataset Paths


In [ ]:
BASE_MODEL_NAME = 'Musixmatch/umberto-commoncrawl-cased-v1'
FORCE_RETRAIN = False  # Set True to retrain even if saved checkpoints exist in ./checkpoints/

from pathlib import Path
repo_root = Path.cwd().resolve()
for _ in range(5):
    if (repo_root / 'datasets').exists():
        break
    repo_root = repo_root.parent

DATASETS_DIR = repo_root / 'datasets' / 'corpora'
MULTIEMIT_TEXTUAL_PATH = str(DATASETS_DIR / 'correctly_split' / 'textual_only_cleaned.csv')
IRONITA_TRAIN_PATH = str(DATASETS_DIR / 'ironita' / 'ironita_full.xlsx')

PRETRAINED_IRONITA_DIR = './checkpoints/irony_textual_ironita_pretrained'
OUTPUT_DIR_NO_CORR = './checkpoints/irony_textual_no_corr'
OUTPUT_DIR_CORR = './checkpoints/irony_textual_corr'

IRONY_LABELS = ['None', 'Ironia', 'Sarcasmo']
id2label = {idx: label for idx, label in enumerate(IRONY_LABELS)}
label2id = {label: idx for idx, label in enumerate(IRONY_LABELS)}
print('Single-Label 3-Class Irony Labels:', IRONY_LABELS)

## 2. Tokenization & Single-Label Evaluation Utilities


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

def preprocess_singlelabel(examples):
    encoding = tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128,
    )
    encoding['labels'] = examples['label']
    return encoding

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_micro': f1_score(labels, preds, average='micro', zero_division=0),
        'f1_weighted': f1_score(labels, preds, average='weighted', zero_division=0),
    }


## 3. Stage 0: Shared Base Pre-training on `IronITA` (4,849 Posts)
Pre-trains `UmBERTo` on `IronITA` so both Stage 1A (Without Correlations) and Stage 1B (With Correlations) start from the **exact same 3-class checkpoint** and are fine-tuned on the **exact same 2,075 MultiEMit training examples**.


In [5]:
df_ironita_raw = pd.read_excel(IRONITA_TRAIN_PATH)
df_ironita_raw['None'] = ((df_ironita_raw['irony'] == 0) & (df_ironita_raw['sarcasm'] == 0)).astype(int)
df_ironita_raw['Ironia'] = ((df_ironita_raw['irony'] == 1) & (df_ironita_raw['sarcasm'] == 0)).astype(int)
df_ironita_raw['Sarcasmo'] = (df_ironita_raw['sarcasm'] == 1).astype(int)

def row_to_irony_class(row):
    if int(row['Sarcasmo']) == 1:
        return 2
    elif int(row['Ironia']) == 1:
        return 1
    return 0

df_ironita_raw['label'] = df_ironita_raw.apply(row_to_irony_class, axis=1)
df_ironita = df_ironita_raw[['text', 'label']].dropna(subset=['text']).reset_index(drop=True)

iron_train_df, iron_val_df = train_test_split(
    df_ironita, test_size=0.1, random_state=SEED, stratify=df_ironita['label']
)
print(f'IronITA Pre-training Split -> Train: {len(iron_train_df)}, Val: {len(iron_val_df)}')

ds_ironita = DatasetDict({
    'train': Dataset.from_pandas(iron_train_df.reset_index(drop=True)),
    'val': Dataset.from_pandas(iron_val_df.reset_index(drop=True)),
})
encoded_ironita = ds_ironita.map(preprocess_singlelabel, batched=True, remove_columns=ds_ironita['train'].column_names)
encoded_ironita.set_format('torch')


IronITA Pre-training Split -> Train: 4364, Val: 485


Map:   0%|          | 0/4364 [00:00<?, ? examples/s]

Map:   0%|          | 0/485 [00:00<?, ? examples/s]

In [ ]:
if os.path.exists(os.path.join(PRETRAINED_IRONITA_DIR, 'config.json')) and not FORCE_RETRAIN:
    print(f'Found existing checkpoint at {PRETRAINED_IRONITA_DIR} — skipping Stage 0 training.')
else:
    model_ironita_base = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_NAME,
        problem_type='single_label_classification',
        num_labels=len(IRONY_LABELS),
        id2label=id2label,
        label2id=label2id,
    )

    args_ironita = TrainingArguments(
        output_dir=PRETRAINED_IRONITA_DIR,
        evaluation_strategy = "epoch",
        save_strategy = "epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=10,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        report_to='none'
    )

    trainer_ironita = Trainer(
        model=model_ironita_base,
        args=args_ironita,
        train_dataset=encoded_ironita['train'],
        eval_dataset=encoded_ironita['val'],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer_ironita.train()
    trainer_ironita.save_model(PRETRAINED_IRONITA_DIR)
    tokenizer.save_pretrained(PRETRAINED_IRONITA_DIR)
    print(f'Saved shared IronITA-pretrained base model to: {PRETRAINED_IRONITA_DIR}')


Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at Musixmatch/umberto-commoncrawl-cased-v1 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\andre\AppData\Local\Temp\ipykernel_7120\1620030846.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_ironita = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,No log,0.844510,0.600000,0.503779,0.568696
2,0.829100,0.859860,0.618557,0.535010,0.594670
3,0.829100,0.885817,0.608247,0.556856,0.605000
4,0.561200,0.977108,0.618557,0.554419,0.606681
5,0.561200,1.021930,0.614433,0.550280,0.603596


Saved shared IronITA-pretrained base model to: ./checkpoints/irony_textual_ironita_pretrained


## 4. Load `MultiEMit_textual_no_incomprensibile.csv` (2,594 Posts) & Build Correlation Prompts
Both Stage 1A and Stage 1B use the **exact same 80/20 split** (`test_size=0.2, random_state=42` -> 2,075 train / 519 test).


In [7]:
df_multiemit = pd.read_csv(MULTIEMIT_TEXTUAL_PATH)
df_multiemit['label'] = df_multiemit.apply(row_to_irony_class, axis=1)

def build_irony_corr_text(df_in):
    """
    Exact correlation prefix generator from italian_correlations.py (target=True, hate=True, emot=True).
    Matches irony_multimodal_corr_ita.csv word-for-word.
    """
    emotion_mapping = {
        'Amore': 'amore',
        'Anticipazione': 'anticipazione',
        'Disgusto': 'disgusto',
        'Fiducia': 'fiducia',
        'Gioia': 'gioia',
        'Paura': 'paura',
        'Rabbia': 'rabbia',
        'Sorpresa': 'sorpresa',
        'Tristezza': 'tristezza',
        'Neutro': 'neutro',
    }
    out_texts = []
    for _, row in df_in.iterrows():
        targets = []
        ironyhate = []
        emotions = []

        if int(row.get('Contenuto', 0)) == 1:
            targets.append('contenuto')
        if int(row.get('Contenitore', 0)) == 1:
            targets.append('contenitore')

        if int(row.get('Offensivita', 0)) == 1:
            ironyhate.append('offensivita')

        for emo_col, emo_it in emotion_mapping.items():
            if int(row.get(emo_col, 0)) == 1:
                emotions.append(emo_it)

        text = str(row['text'])
        if targets:
            text = 'riguarda {} '.format(' e '.join(targets)) + text
        if ironyhate:
            text = 'contiene {} '.format(' e '.join(ironyhate)) + text
        if emotions:
            text = 'esprime {} '.format(' e '.join(emotions)) + text

        text = 'il seguente testo ' + text
        out_texts.append(text)
    return out_texts

df_multiemit['text_corr'] = build_irony_corr_text(df_multiemit)

me_train, me_test = train_test_split(df_multiemit, test_size=0.2, random_state=SEED)
print(f'MultiEMit Split -> Train: {len(me_train)}, Test: {len(me_test)}')
print('Sample Raw Text:       ', me_train['text'].iloc[0][:100])
print('Sample Correlated Text:', me_train['text_corr'].iloc[0][:140])


MultiEMit Split -> Train: 2075, Test: 519
Sample Raw Text:        secondo me hanno delle possibilita di vittoria asia e vera 😂 #pechinoexpress
Sample Correlated Text: il seguente testo presenta l'emozione sorpresa e si riferisce a contenitore: secondo me hanno delle possibilita di vittoria asia e vera 😂 #p


## 5. Stage 1A: Fine-Tune (or Load) on MultiEMit **Without Correlations**
Starts from `./checkpoints/irony_textual_ironita_pretrained` and fine-tunes on MultiEMit raw text (or loads `./checkpoints/irony_textual_no_corr` directly if already trained).


In [ ]:
train_no_corr = me_train[['text', 'label']].reset_index(drop=True)
test_no_corr = me_test[['text', 'label']].reset_index(drop=True)

ds_no_corr = DatasetDict({
    'train': Dataset.from_pandas(train_no_corr),
    'test': Dataset.from_pandas(test_no_corr),
})
encoded_no_corr = ds_no_corr.map(preprocess_singlelabel, batched=True, remove_columns=ds_no_corr['train'].column_names)
encoded_no_corr.set_format('torch')

args_no_corr = TrainingArguments(
    output_dir=OUTPUT_DIR_NO_CORR,
    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    report_to='none'
)

if os.path.exists(os.path.join(OUTPUT_DIR_NO_CORR, 'config.json')) and not FORCE_RETRAIN:
    print(f'Loading existing checkpoint from {OUTPUT_DIR_NO_CORR} (skipping training)...')
    model_no_corr = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR_NO_CORR)
    trainer_no_corr = Trainer(
        model=model_no_corr,
        args=args_no_corr,
        eval_dataset=encoded_no_corr['test'],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )
else:
    model_no_corr = AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED_IRONITA_DIR,
        problem_type='single_label_classification',
        num_labels=len(IRONY_LABELS),
        id2label=id2label,
        label2id=label2id,
    )
    trainer_no_corr = Trainer(
        model=model_no_corr,
        args=args_no_corr,
        train_dataset=encoded_no_corr['train'],
        eval_dataset=encoded_no_corr['test'],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )
    trainer_no_corr.train()
    trainer_no_corr.save_model(OUTPUT_DIR_NO_CORR)
    tokenizer.save_pretrained(OUTPUT_DIR_NO_CORR)

preds_no_corr = trainer_no_corr.predict(encoded_no_corr['test'])
y_pred_no_corr = np.argmax(preds_no_corr.predictions, axis=-1)
y_true_no_corr = preds_no_corr.label_ids

print('=== Textual Irony Detection (WITHOUT Correlations) — MultiEMit Test Set ===')
print(classification_report(y_true_no_corr, y_pred_no_corr, target_names=IRONY_LABELS, digits=4, zero_division=0))


Map:   0%|          | 0/2075 [00:00<?, ? examples/s]

Map:   0%|          | 0/519 [00:00<?, ? examples/s]

C:\Users\andre\AppData\Local\Temp\ipykernel_7120\994806575.py:45: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_no_corr = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,No log,0.485360,0.805395,0.372503,0.750269
2,No log,0.518373,0.793834,0.433008,0.777008
3,No log,0.639909,0.793834,0.411802,0.766283
4,0.380400,0.748910,0.782274,0.418846,0.765219
5,0.380400,0.769147,0.780347,0.417678,0.763748


=== Textual Irony Detection (WITHOUT Correlations) — MultiEMit Test Set ===
              precision    recall  f1-score   support

        None     0.8446    0.9102    0.8762       412
      Ironia     0.4933    0.3700    0.4229       100
    Sarcasmo     0.0000    0.0000    0.0000         7

    accuracy                         0.7938       519
   macro avg     0.4460    0.4267    0.4330       519
weighted avg     0.7655    0.7938    0.7770       519



## 6. Stage 1B: Fine-Tune (or Load) on MultiEMit **With Correlations**
Starts from the **exact same** `./checkpoints/irony_textual_ironita_pretrained` checkpoint and fine-tunes on the **exact same** MultiEMit split using the full correlation prefixes (or loads `./checkpoints/irony_textual_corr` directly if already trained).


In [ ]:
train_corr = me_train[['text_corr', 'label']].rename(columns={'text_corr': 'text'}).reset_index(drop=True)
test_corr = me_test[['text_corr', 'label']].rename(columns={'text_corr': 'text'}).reset_index(drop=True)

ds_corr = DatasetDict({
    'train': Dataset.from_pandas(train_corr),
    'test': Dataset.from_pandas(test_corr),
})
encoded_corr = ds_corr.map(preprocess_singlelabel, batched=True, remove_columns=ds_corr['train'].column_names)
encoded_corr.set_format('torch')

args_corr = TrainingArguments(
    output_dir=OUTPUT_DIR_CORR,
    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    report_to='none'
)

if os.path.exists(os.path.join(OUTPUT_DIR_CORR, 'config.json')) and not FORCE_RETRAIN:
    print(f'Loading existing checkpoint from {OUTPUT_DIR_CORR} (skipping training)...')
    model_corr = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR_CORR)
    trainer_corr = Trainer(
        model=model_corr,
        args=args_corr,
        eval_dataset=encoded_corr['test'],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )
else:
    model_corr = AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED_IRONITA_DIR,
        problem_type='single_label_classification',
        num_labels=len(IRONY_LABELS),
        id2label=id2label,
        label2id=label2id,
    )
    trainer_corr = Trainer(
        model=model_corr,
        args=args_corr,
        train_dataset=encoded_corr['train'],
        eval_dataset=encoded_corr['test'],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )
    trainer_corr.train()
    trainer_corr.save_model(OUTPUT_DIR_CORR)
    tokenizer.save_pretrained(OUTPUT_DIR_CORR)

preds_corr = trainer_corr.predict(encoded_corr['test'])
y_pred_corr = np.argmax(preds_corr.predictions, axis=-1)
y_true_corr = preds_corr.label_ids

print('=== Textual Irony Detection (WITH Correlations) — MultiEMit Test Set ===')
print(classification_report(y_true_corr, y_pred_corr, target_names=IRONY_LABELS, digits=4, zero_division=0))


Map:   0%|          | 0/2075 [00:00<?, ? examples/s]

Map:   0%|          | 0/519 [00:00<?, ? examples/s]

C:\Users\andre\AppData\Local\Temp\ipykernel_7120\2852056572.py:45: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_corr = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,No log,0.460209,0.784200,0.368746,0.740762


KeyboardInterrupt: 

## 7. Summary Comparison Table (Without vs. With Correlations)


In [ ]:
os.makedirs('./results', exist_ok=True)

rep_no_corr = classification_report(y_true_no_corr, y_pred_no_corr, target_names=IRONY_LABELS, digits=4, zero_division=0)
rep_corr = classification_report(y_true_corr, y_pred_corr, target_names=IRONY_LABELS, digits=4, zero_division=0)

print('=== Textual Irony Detection (WITHOUT Correlations) — MultiEMit Test Set ===')
print(rep_no_corr)

print('\n=== Textual Irony Detection (WITH Correlations) — MultiEMit Test Set ===')
print(rep_corr)

summary_df = pd.DataFrame([
    {
        'Task': 'Irony & Sarcasm Detection (3-Class Single-Label)',
        'Modality': 'Textual',
        'Notebook': 'train_textual_irony.ipynb',
        'Configuration': 'UmBERTo (IronITA Pre-trained -> MultiEMit No Correlations)',
        'Micro F1': f1_score(y_true_no_corr, y_pred_no_corr, average='micro', zero_division=0),
        'Macro F1': f1_score(y_true_no_corr, y_pred_no_corr, average='macro', zero_division=0),
        'Weighted F1': f1_score(y_true_no_corr, y_pred_no_corr, average='weighted', zero_division=0),
        'Accuracy / Subset Acc': accuracy_score(y_true_no_corr, y_pred_no_corr),
    },
    {
        'Task': 'Irony & Sarcasm Detection (3-Class Single-Label)',
        'Modality': 'Textual',
        'Notebook': 'train_textual_irony.ipynb',
        'Configuration': 'UmBERTo (IronITA Pre-trained -> MultiEMit With Correlations)',
        'Micro F1': f1_score(y_true_corr, y_pred_corr, average='micro', zero_division=0),
        'Macro F1': f1_score(y_true_corr, y_pred_corr, average='macro', zero_division=0),
        'Weighted F1': f1_score(y_true_corr, y_pred_corr, average='weighted', zero_division=0),
        'Accuracy / Subset Acc': accuracy_score(y_true_corr, y_pred_corr),
    }
])

summary_df.to_csv('./results/train_textual_irony_summary.csv', index=False)
with open('./results/train_textual_irony_reports.txt', 'w', encoding='utf-8') as f_rep:
    f_rep.write('=== Textual Irony Detection (WITHOUT Correlations) ===\n' + rep_no_corr + '\n\n')
    f_rep.write('=== Textual Irony Detection (WITH Correlations) ===\n' + rep_corr + '\n')

display(summary_df.round(4))


=== Textual Irony Detection (WITHOUT Correlations) — MultiEMit Test Set ===
              precision    recall  f1-score   support

        None     0.8446    0.9102    0.8762       412
      Ironia     0.4933    0.3700    0.4229       100
    Sarcasmo     0.0000    0.0000    0.0000         7

    accuracy                         0.7938       519
   macro avg     0.4460    0.4267    0.4330       519
weighted avg     0.7655    0.7938    0.7770       519


=== Textual Irony Detection (WITH Correlations) — MultiEMit Test Set ===
              precision    recall  f1-score   support

        None     0.8598    0.8932    0.8762       412
      Ironia     0.4615    0.4200    0.4398       100
    Sarcasmo     0.0000    0.0000    0.0000         7

    accuracy                         0.7900       519
   macro avg     0.4405    0.4377    0.4387       519
weighted avg     0.7715    0.7900    0.7803       519



,Configuration,Accuracy,Macro F1,Micro F1,Weighted F1
0,UmBERTo (IronITA Pre-trained -> MultiEMit No C...,0.7938,0.4330,0.79,0.7770
1,UmBERTo (IronITA Pre-trained -> MultiEMit With...,0.7900,0.4387,0.79,0.7803
